# QC/trimming samples 
- (sequenced Jan 2024)
- PSTR, OFAV, OANN, MCAV, MMEA
- 2019 and 2022 

In [ ]:
# unzip files, make sample list, rename files in dir

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=50G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs012024/unziplist%j.out  # %j = job ID

# unzip files
cd /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/raw

# unzip 
gunzip *.fastq.gz

## Get sample list 
ls | head -n 1| cut -d '_' -f1-8 | sed 's/_S[0-9]\+_R[12]//' | sort -u > /scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/sampleids.txt

# remove sequencer ID from file names 
for file in *; do
    new_name=$(echo "$file" | sed 's/_S[0-9]\+//')
    mv "$file" "$new_name"
done

# job id- 51476024

In [ ]:
# qc

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=50G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/brooke/seqs012024/slurm-qc-%j.out  # %j = job ID

module load conda/latest
conda activate qc

# Define the paths and variables
FILEPATH='/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/raw'
OUTPUT_RESULTS='/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/trimmed' 
NSLOTS=4  
SAMPLE_NAMES_FILE="/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw/sampleids.txt"

# Check if the file exists
if [ ! -e "$SAMPLE_NAMES_FILE" ]; then
    echo "Error: $SAMPLE_NAMES_FILE does not exist."
    exit 1
fi

# Read each line from the file and perform actions
while IFS= read -r sample_id; do
    # Form the full file names
    input_r1="$FILEPATH/${sample_id}_R1_001.fastq.gz"
    input_r2="$FILEPATH/${sample_id}_R2_001.fastq.gz"
    
    # Ensure the input files exist before running the tools
    if [ ! -e "$input_r1" ] || [ ! -e "$input_r2" ]; then
        echo "Error: Input files do not exist for sample $sample_id"
        continue
    fi

    # Run trim_galore
    trim_galore -j "$NSLOTS" -q 20 --phred33 --length 20 --paired $input_r1 $input_r2 --fastqc -o $OUTPUT_RESULTS --dont_gzip
#forgot to include --fastqc..running after

done < "$SAMPLE_NAMES_FILE"

# JOB-ID: 18980369
# script file: /project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/11_2023bash_scripts/qc
#trimmed read seqs in folder: /project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/working/11272023/trimmed/redo_02072024